# 기존 MLP 새 커널 재현 — CUDA 작업 사본

현재 `main.ipynb`에 저장된 기존 코드와 학습자의 장치 수정 내용을 복사했다.
데이터 준비·모델 정의는 AI가 제공했고, 학습·평가 함수는 학습자가 작성하며 API 도움을 받은 코드다.
이번 실습은 기존 흐름을 새 커널에서 실행하고 해석하는 것이다.

원본의 코드 셀은 변경하지 않았고, 이 작업 사본의 이전 출력과 실행 번호만 제거했다.
원본 `main.ipynb`와 기존 보관본은 보존했다. 작업 사본은 아직 실행하지 않았다.

커널은 `Python 3.14 (llm-learning-lab)`을 사용한다.
프로젝트 환경에서 CUDA 사용 가능 여부와 별도 GPU 텐서·행렬곱 점검을 확인했다.
이는 아래 노트북의 실행이나 전체 재현 성공을 뜻하지 않는다.

먼저 `import random`과 `def build_dataset(word_list):`로 시작하는 코드 셀 두 개를 직접 실행하고 저장한다.
입력·정답 shape와 각 축의 의미, 선택된 device를 실제 출력으로 설명한다.
그 다음 모델·학습·평가를 단계별로 재현한다.
학습 호출 전 난수 생성기·배치 인덱스·데이터·모델의 장치 설정을 확인하고,
필요한 학습 코드 수정은 학습자가 직접 수행한다.

- [이번 복사 원본](../../main.ipynb)
- [기존 보관본](./makemore-mlp-training-recall.ipynb)
- [기존 수행과 도움 범위](./makemore-mlp-training-recall.md)
- [공식 강의: Building makemore Part 2: MLP](https://www.youtube.com/watch?v=TCH_1BHY58I)
- [데이터 출처: Karpathy makemore names.txt](https://github.com/karpathy/makemore/blob/master/names.txt)


In [1]:
import random
from urllib.request import urlopen

import torch
import torch.nn.functional as F

with urlopen("https://raw.githubusercontent.com/karpathy/makemore/master/names.txt", timeout=30) as response:
    words = response.read().decode("utf-8").splitlines()

chars = sorted(set("".join(words)))
stoi = {ch: i + 1 for i, ch in enumerate(chars)}
stoi["."] = 0
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(stoi)
block_size = 3
embedding_dim = 10
hidden_size = 200
batch_size = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("words:", len(words), "vocabulary:", vocab_size, "device:", device)

words: 32033 vocabulary: 27 device: cuda


In [2]:
def build_dataset(word_list):
    X, Y = [], []
    for word in word_list:
        context = [0] * block_size
        for ch in word + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return (
        torch.tensor(X, dtype=torch.long, device=device),
        torch.tensor(Y, dtype=torch.long, device=device),
    )

shuffled_words = words.copy()
random.Random(42).shuffle(shuffled_words)
n = len(shuffled_words)
train_end, dev_end = n * 8 // 10, n * 9 // 10
Xtr, Ytr = build_dataset(shuffled_words[:train_end])
Xdev, Ydev = build_dataset(shuffled_words[train_end:dev_end])
Xte, Yte = build_dataset(shuffled_words[dev_end:])
for split_name, split_X, split_Y in [("train", Xtr, Ytr), ("dev", Xdev, Ydev), ("test", Xte, Yte)]:
    print(split_name, split_X.shape, split_Y.shape)
del split_name, split_X, split_Y

train torch.Size([182625, 3]) torch.Size([182625])
dev torch.Size([22655, 3]) torch.Size([22655])
test torch.Size([22866, 3]) torch.Size([22866])


In [3]:
# 재실행하면 가중치와 난수 시퀀스를 처음 상태로 되돌립니다.
g_model = torch.Generator(device=device).manual_seed(2147483647)
g_batch = torch.Generator(device=device).manual_seed(42)
C = torch.randn((vocab_size, embedding_dim), generator=g_model, device=device, dtype=torch.float32)
W1 = torch.randn((block_size * embedding_dim, hidden_size), generator=g_model, device=device, dtype=torch.float32)
b1 = torch.randn(hidden_size, generator=g_model, device=device, dtype=torch.float32)
W2 = torch.randn((hidden_size, vocab_size), generator=g_model, device=device, dtype=torch.float32) * 0.02
b2 = torch.zeros(vocab_size, device=device, dtype=torch.float32)
W_direct = torch.zeros((block_size * embedding_dim, vocab_size), dtype=W1.dtype, device=W1.device)
parameters_direct = [C, W1, b1, W2, b2, W_direct]
for p in parameters_direct:
    p.requires_grad_(True)

print("parameters:", sum(p.numel() for p in parameters_direct))


parameters: 12707


In [4]:
def forward_direct(X):
    emb = C[X]
    flat = emb.flatten(start_dim=1)
    direct_scores = flat @ W_direct
    h = torch.tanh(flat @ W1 + b1)
    return h @ W2 + b2 + direct_scores

## 기존 학습·평가 구현

아래는 학습자가 이전에 작성한 함수다. 이번에는 기존 구현의 실행과 결과 해석을 확인한다.
학습 호출 전에 현재 장치와 함수의 장치 설정을 확인한다.

준비된 객체:

| 이름 | 용도 |
|---|---|
| `Xtr`, `Ytr` | 학습 입력과 정답 |
| `Xdev`, `Ydev` | 검증 입력과 정답 |
| `Xte`, `Yte` | 최종 평가용 데이터. 이번 구현·튜닝에는 사용하지 않음 |
| `forward_direct` | 입력에서 logits를 계산하는 직접 연결 MLP |
| `parameters_direct` | 모델의 학습 가능한 가중치 목록 |
| `batch_size` | 기존 실험의 배치 크기 |
| `g_batch` | 배치 선택에 사용할 난수 생성기 |
| `device` | 현재 실행 장치 |

아래 학습 호출은 학습률 0.03, 1,000회이며, 이 새 커널에서의 결과는 아직 확인하지 않았다.


In [8]:
def train_step_direct(iter, lr):
    for step in range(iter):
        indices = torch.randint(high=len(Xtr), size=(batch_size,), generator=g_batch, device="cuda")
        loss = F.cross_entropy(forward_direct(Xtr[indices]), Ytr[indices])

        for p in parameters_direct:
            p.grad = None

        loss.backward()

        with torch.no_grad():
            for p in parameters_direct:
                p -= p.grad * lr

@torch.no_grad()
def eval_direct(X, Y):
    return F.cross_entropy(forward_direct(X), Y)

In [10]:
train_loss = eval_direct(Xtr, Ytr).item()
dev_loss = eval_direct(Xdev, Ydev).item()

print(f"train_loss: {train_loss}")
print(f"dev_loss: {dev_loss}")

train_loss: 3.348036527633667
dev_loss: 3.348778486251831


In [11]:
lr = 0.03
iter = 1000

train_step_direct(iter, lr)

In [12]:
train_loss = eval_direct(Xtr, Ytr).item()
dev_loss = eval_direct(Xdev, Ydev).item()

print(f"train_loss: {train_loss}")
print(f"dev_loss: {dev_loss}")

train_loss: 2.4138410091400146
dev_loss: 2.4144763946533203
